In [1]:
import polars as p
import polars as pl
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import classification_report, average_precision_score, precision_recall_curve

df = pl.read_csv('creditcard.csv', infer_schema_length=1000000)

features_reg = [f'V{i}' for i in range(1, 29)] + ['Time']
target_reg = 'Amount'

print('обучение моделей ураааааааааааааааааааа')

df_legal = df.filter(pl.col('Class') == 0)

X_reg = df_legal.select(features_reg).to_pandas()
y_reg = df_legal.select(target_reg).to_pandas().values.ravel()

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

models_reg = {
    'Ridge Regression': Ridge(alpha=1.0), # базовая линейная зависимость
    'Random Forest Regressor': RandomForestRegressor(n_estimators=30, max_depth=8, random_state=1984, n_jobs=-1), # ступенчатая аппроксимация
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=50, max_depth=5, random_state=1984) # последовательное исправление ошибок
}

results_reg = {}
feature_importances = {}

for name, model in models_reg.items():
    model.fit(X_train_r, y_train_r)
    preds = model.predict(X_test_r)
    
    rmse = np.sqrt(mean_squared_error(y_test_r, preds))
    mae = mean_absolute_error(y_test_r, preds)
    r2 = r2_score(y_test_r, preds)
    
    results_reg[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    
    if name == 'Ridge Regression':
        importances = np.abs(model.coef_)
    else:
        importances = model.feature_importances_
        
    imp_df = pl.DataFrame({
        'Feature': features_reg,
        'Importance': importances
    }).sort('Importance', descending=True)
    
    feature_importances[name] = imp_df

print('\nрезультаты моделей регрессии:')
for name, metrics in results_reg.items():
    print(f'{name}: RMSE = {metrics["RMSE"]:.4f} | MAE = {metrics["MAE"]:.4f} | R2 = {metrics["R2"]:.4f}')

print('\nтоп-5 важных признаков для каждой модели регрессии:')
for name, imp_df in feature_importances.items():
    print(f'\n[{name}]')
    print(imp_df.head(5))


print('\nклассификация')

X_clf = df.select(features_reg + ['Amount']).to_pandas()
y_clf = df.select('Class').to_pandas().values.ravel()

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.3, stratify=y_clf, random_state=42
)

clf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

print('обучение модели классификации random forest ваууууу')
clf_model.fit(X_train_c, y_train_c)

preds_proba = clf_model.predict_proba(X_test_c)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test_c, preds_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f'оптимальный порог классификации: {best_threshold:.4f}')

preds_binary = (preds_proba >= best_threshold).astype(int)

print('\nотчет о классификации')
print(classification_report(y_test_c, preds_binary))

pr_auc = average_precision_score(y_test_c, preds_proba)
print(f'precision-recall auc: {pr_auc:.4f}')

best_model_name = max(results_reg, key=lambda k: results_reg[k]['R2'])
best_metrics = results_reg[best_model_name]

print(f'лучшая модель регрессии: {best_model_name}')
print(f'критерий выбора: максимальный r2 = {best_metrics["R2"]:.4f}')
print(f'сопутствующие ошибки: rmse = {best_metrics["RMSE"]:.4f} | mae = {best_metrics["MAE"]:.4f}')

обучение моделей ураааааааааааааааааааа



результаты моделей регрессии:
Ridge Regression: RMSE = 78.7305 | MAE = 23.5255 | R2 = 0.9033
Random Forest Regressor: RMSE = 87.3568 | MAE = 33.3252 | R2 = 0.8810
Gradient Boosting Regressor: RMSE = 76.8901 | MAE = 26.1680 | R2 = 0.9078

топ-5 важных признаков для каждой модели регрессии:

[Ridge Regression]
shape: (5, 2)
┌─────────┬────────────┐
│ Feature ┆ Importance │
│ ---     ┆ ---        │
│ str     ┆ f64        │
╞═════════╪════════════╡
│ V20     ┆ 113.319541 │
│ V2      ┆ 80.548852  │
│ V7      ┆ 79.120066  │
│ V5      ┆ 68.278627  │
│ V23     ┆ 48.190902  │
└─────────┴────────────┘

[Random Forest Regressor]
shape: (5, 2)
┌─────────┬────────────┐
│ Feature ┆ Importance │
│ ---     ┆ ---        │
│ str     ┆ f64        │
╞═════════╪════════════╡
│ V7      ┆ 0.284416   │
│ V20     ┆ 0.270047   │
│ V2      ┆ 0.243145   │
│ V5      ┆ 0.093829   │
│ V23     ┆ 0.02761    │
└─────────┴────────────┘

[Gradient Boosting Regressor]
shape: (5, 2)
┌─────────┬────────────┐
│ Feature ┆ Im

обучение модели классификации random forest ваууууу


оптимальный порог классификации: 0.5458

отчет о классификации
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.85      0.76      0.80       148

    accuracy                           1.00     85443
   macro avg       0.93      0.88      0.90     85443
weighted avg       1.00      1.00      1.00     85443

precision-recall auc: 0.7839
лучшая модель регрессии: Gradient Boosting Regressor
критерий выбора: максимальный r2 = 0.9078
сопутствующие ошибки: rmse = 76.8901 | mae = 26.1680
